# PoC - Modelo de Status de Operacao do Onibus

Este notebook cria uma prova de conceito para classificar o `status_operacao` do onibus com base nos dados publicados pelo produtor.

- Dataset de origem: `producer/bus/dataset/tempo_real_convencional_json_070326052133.json`
- Abordagem: gerar um alvo simples (`parado`, `lento`, `normal`) a partir de `VL` e treinar um classificador com os demais campos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path('../../../producer/bus/dataset/tempo_real_convencional_json_070326052133.json').resolve()
MODEL_PATH = Path('bus_status_model.joblib').resolve()

DATA_PATH, MODEL_PATH

(PosixPath('/home/makleyston/Projects/service-composition/producer/bus/dataset/tempo_real_convencional_json_070326052133.json'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-operation-status/bus_status_model.joblib'))

In [2]:
df = pd.read_json(DATA_PATH)
df.head()

,EV,HR,LT,LG,NV,VL,NL,DG,SV,DT
0,105,20260307171421,-19.797320,-44.013542,31138,0,7979.0,69,0.0,5400.0
1,105,20260307171653,-19.828992,-43.915084,21142,15,6760.0,183,1.0,3332.0
2,105,20260307164449,-19.909361,-43.894718,20884,0,629.0,0,0.0,1487.0
3,105,20260307171656,-19.953643,-43.969597,30915,0,97.0,56,0.0,2469.0
4,105,20260307171656,-19.789555,-44.015885,11281,0,5577.0,229,NaN,9051.0


In [3]:
# Conversao de tipos
for col in ['LT', 'LG', 'VL', 'DG', 'DT']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['SV', 'NL', 'NV']:
    df[col] = df[col].astype('object')

df['timestamp'] = pd.to_datetime(df['HR'], format='%Y%m%d%H%M%S', errors='coerce')
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.dayofweek

# Rotulo simples para PoC (regra baseada em velocidade)
conditions = [
    df['VL'] <= 2,
    (df['VL'] > 2) & (df['VL'] <= 20),
    df['VL'] > 20,
]
choices = ['parado', 'lento', 'normal']
df['status_operacao'] = np.select(conditions, choices, default='indefinido')

df = df[df['status_operacao'] != 'indefinido'].copy()
df['status_operacao'].value_counts(normalize=True).rename('proporcao')

status_operacao
parado    0.784258
normal    0.110130
lento     0.105612
Name: proporcao, dtype: float64

In [4]:
# Nao usamos VL no treino para evitar vazamento direto do rotulo
feature_cols = ['LT', 'LG', 'DG', 'SV', 'DT', 'hour', 'weekday', 'NL']
target_col = 'status_operacao'

X = df[feature_cols]
y = df[target_col]

numeric_features = ['LT', 'LG', 'DG', 'DT', 'hour', 'weekday']
categorical_features = ['SV', 'NL']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
print('Matriz de confusao:')
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

       lento       0.51      0.51      0.51       603
      normal       0.60      0.58      0.59       629
      parado       0.96      0.97      0.97      4478

    accuracy                           0.88      5710
   macro avg       0.69      0.68      0.69      5710
weighted avg       0.87      0.88      0.88      5710

Matriz de confusao:
[[ 306  180  117]
 [ 214  362   53]
 [  76   66 4336]]


In [5]:
preview = X_test.head(5).copy()
preview['status_real'] = y_test.head(5).values
preview['status_predito'] = clf.predict(X_test.head(5))
preview

,LT,LG,DG,SV,DT,hour,weekday,NL,status_real,status_predito
17247,-19.789762,-44.015302,301,0.0,8556.0,17,5,548.0,parado,parado
8882,-19.885165,-43.979913,0,0.0,7370.0,17,5,345.0,parado,parado
22255,-19.899117,-43.911970,17,2.0,12739.0,17,5,565.0,normal,lento
9828,-19.801062,-43.978387,79,NaN,3597.0,17,5,1920.0,parado,parado
3941,-19.797727,-43.953518,74,1.0,10635.0,17,5,3896.0,parado,parado


In [6]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'labeling_rule': {
        'parado': 'VL <= 2',
        'lento': '2 < VL <= 20',
        'normal': 'VL > 20',
    },
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH

PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-operation-status/bus_status_model.joblib')